In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from typing import List, Tuple

class GeneNetworkAnalyzer:
    """分析基因交互網路並計算相關性數值 (RV) 的工具類別"""
    
    def __init__(self, alpha: float = 1.0):
        self.alpha = alpha
        self.model = Ridge(alpha=self.alpha, fit_intercept=False)

    def solve_interaction_matrix(self, X: np.ndarray) -> np.ndarray:
        """
        使用逐行 Ridge Regression 求解基因互動矩陣 A。
        A[i, j] 代表基因 j 對基因 i 的影響力，且對角線 A[i, i] = 0。
        """
        n_samples, p_genes = X.shape
        A = np.zeros((p_genes, p_genes))

        for i in range(p_genes):
            # 目標：第 i 個基因的表現量
            y = X[:, i]
            # 特徵：排除第 i 個基因後的其餘基因
            X_features = np.delete(X, i, axis=1)
            
            # 訓練模型
            self.model.fit(X_features, y)
            
            # 將係數填回矩陣，並確保對角線為 0
            A[i, :] = np.insert(self.model.coef_, i, 0)
            
        return A

    def calculate_relevance_values(self, A_pos: np.ndarray, A_neg: np.ndarray, gene_names: List[str]) -> pd.DataFrame:
        """計算 RV 並返回排序後的 DataFrame"""
        # RV 為 A_diff 矩陣中每一行絕對值的總和 (對應 column sum)
        rv_scores = np.sum(np.abs(A_pos - A_neg), axis=0)
        
        df_rv = pd.DataFrame({
            'Gene': gene_names,
            'Relevance_Value': rv_scores
        })
        return df_rv.sort_values(by='Relevance_Value', ascending=False).reset_index(drop=True)

# --- 執行流程 ---
if __name__ == "__main__":
    
    data = np.load('ANOVA_grouped_CADM1.npz', allow_pickle=True)
    X_pos, X_neg, G = data['x_pos'], data['x_neg'], data['ANOVA_genes']

    analyzer = GeneNetworkAnalyzer(alpha=1.0)
    
    print("Solving for A+ and A- matrices...")
    A_pos = analyzer.solve_interaction_matrix(X_pos)
    A_neg = analyzer.solve_interaction_matrix(X_neg)
    
    print("Calculating Relevance Values...")
    rv_results = analyzer.calculate_relevance_values(A_pos, A_neg, G)
    
    print("\nTop 15 Genes:")
    print(rv_results.head(15))

# --- Data Loading ---
# 載入 .npz 檔案
try:
    # Added allow_pickle=True to load object arrays
    data = np.load('AIIM_HW2_ANOVA_grouped_CADM1.npz', allow_pickle=True)
except FileNotFoundError:
    print("Error: 'AIIM_HW2_ANOVA_grouped_CADM1.npz' not found.")
    print("Please make sure the data file is in the same directory as the script.")
    exit()

# 提取所需的數據
# X+ (CADM1 高表現組)，維度為 (病患數, 基因數)
X_pos = data['x_pos']
# X- (CADM1 低表現組)
X_neg = data['x_neg']
# ANOVA 篩選出的基因列表
G = data['ANOVA_genes']

print("--- Data Information ---")
print(f"Shape of X_pos (Positive Group): {X_pos.shape}")
print(f"Shape of X_neg (Negative Group): {X_neg.shape}")
print(f"Number of ANOVA selected genes: {len(G)}")
print("-" * 26)

# ===================================================================
# Solve for A+ and A- using Ridge Regression
# ===================================================================

def solve_interaction_matrix(X_group):

    # 獲取基因數量 p
    _, p_genes = X_group.shape

    # 初始化 A 矩陣為 p x p 的零矩陣
    A = np.zeros((p_genes, p_genes))

    # 建立 Ridge 迴歸模型
    # alpha=1: L2-regularization coefficient
    # fit_intercept=False: 我們的模型 x_i = sum(a_ij * x_j) 沒有常數項/截距項
    model = Ridge(alpha=1, fit_intercept=False)

    # 逐行 (row-by-row) 求解 A 矩陣
    # 迴圈遍歷每一個基因 i，將其作為迴歸的目標 (target)
    for i in range(p_genes):
        # 目標 y 是第 i 個基因在所有病患的表現量 (X_group 的第 i 個 column)
        y_target = X_group[:, i]

        # 特徵 X_features 是除了第 i 個基因外的所有其他基因的表現量
        # 我們從 X_group 中移除第 i 個 column
        X_features = np.delete(X_group, i, axis=1)

        # 使用特徵和目標來訓練模型
        model.fit(X_features, y_target)

        # 獲取訓練好的模型係數 (coeffs)，這是 A 的第 i 行，但不包含 a_ii
        coeffs = model.coef_

        # 將係數插入 A 的第 i 行。在對角線位置 i 補上 0，因為 a_ii = 0
        A[i, :] = np.insert(coeffs, i, 0)

    return A

print("\n--- Q2(b): Solving for A+ and A- ---")

# 分別求解 Positive group 和 Negative group 的互動矩陣
A_pos = solve_interaction_matrix(X_pos)
A_neg = solve_interaction_matrix(X_neg)

print(f"Successfully solved A+ matrix with shape: {A_pos.shape}")
print(f"Successfully solved A- matrix with shape: {A_neg.shape}")
print("-" * 38)

# ===================================================================
# Calculate Relevance Values (RVs) and list Top-15 genes
# ===================================================================

print("\n--- Q2(c): Calculating RVs and Finding Top-15 Genes ---")

# 1. 計算兩個互動矩陣的差異
A_diff = A_pos - A_neg

# 2. 取差異矩陣的絕對值
A_diff_abs = np.abs(A_diff)

# 3. 對 column 求和 (axis=0)，得到每個基因的 RV
# RV_j = sum_i |a_ij(+) - a_ij(-)|
RVs = np.sum(A_diff_abs, axis=0)

# 4. 將基因名稱和對應的 RVs 配對
gene_rv_pairs = list(zip(G, RVs))

# 5. 根據 RVs 的值進行降冪排序
# 使用 lambda 函數指定排序的鍵 (key) 為元組的第二個元素 (RV值)
sorted_gene_rvs = sorted(gene_rv_pairs, key=lambda item: item[1], reverse=True)

# 6. 列出 Top-15 的基因及其 RVs
print("Top 15 genes by Relevance Value (RV):")
print("=" * 40)
print(f"{'Rank':<5} {'Gene':<10} {'Relevance Value':<20}")
print("-" * 40)
top_15_genes = []
for i, (gene, rv) in enumerate(sorted_gene_rvs[:15]):
    print(f"{i+1:<5} {gene:<10} {rv:.4f}")
    top_15_genes.append(gene)
print("=" * 40)

print("\nList of Top-15 gene names for Q3:")
print(top_15_genes)
print("-" * 40)

--- Data Information ---
Shape of X_pos (Positive Group): (27, 329)
Shape of X_neg (Negative Group): (28, 329)
Number of ANOVA selected genes: 329
--------------------------

--- Q2(b): Solving for A+ and A- ---
Successfully solved A+ matrix with shape: (329, 329)
Successfully solved A- matrix with shape: (329, 329)
--------------------------------------

--- Q2(c): Calculating RVs and Finding Top-15 Genes ---
Top 15 genes by Relevance Value (RV):
Rank  Gene       Relevance Value     
----------------------------------------
1     PLAC8      4.9781
2     MMP9       4.8135
3     FAM198B    4.5533
4     PRICKLE1   4.0398
5     ZFP3       4.0019
6     SNORD89    3.9309
7     LOC202181  3.9125
8     ZNF383     3.8391
9     LOC389765  3.8162
10    S100A16    3.7446
11    NDUFAF2    3.6928
12    SCIN       3.5561
13    ANKRD12    3.5531
14    HOXB2      3.4953
15    ADAMTS5    3.4848

List of Top-15 gene names for Q3:
['PLAC8', 'MMP9', 'FAM198B', 'PRICKLE1', 'ZFP3', 'SNORD89', 'LOC202181', '